In [ ]:
import librosa
import torch
from datasets import load_dataset, DatasetDict, concatenate_datasets
from transformers import WhisperProcessor, WhisperFeatureExtractor
from huggingface_hub import login
import random
import numpy as np

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

import os
login(token=os.environ["HF_TOKEN"])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

main_seed=42
# model_name = "openai/whisper-large-v3"
model_name = "openai/whisper-small"

model_language = "hungarian"

stt_path = "../data/stt_dataset/"
tts_path = "../data/tts_dataset/"
bonus_path = "../data/bonus_dataset/"

In [ ]:
stt_train_dataset = load_dataset('csv', data_files=stt_path+'train/metadata.csv', delimiter='\t', split='train')
stt_test_dataset = load_dataset('csv', data_files=stt_path+'test/metadata.csv', delimiter='\t', split='train')

tts_train_dataset = load_dataset('csv', data_files=tts_path+'train/metadata.csv', delimiter='\t', split='train')
tts_test_dataset = load_dataset('csv', data_files=tts_path+'test/metadata.csv', delimiter='\t', split='train')

bonus_train_dataset = load_dataset('csv', data_files=bonus_path+'train/metadata.csv', delimiter='\t', split='train')
bonus_test_dataset = load_dataset('csv', data_files=bonus_path+'test/metadata.csv', delimiter='\t', split='train')

In [ ]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name)

def augment_audio(y, sr=16000):
    aug_type = random.choice(["stretch", "pitch", "noise"])
    
    if aug_type == "stretch":
        rate = random.uniform(0.9, 1.1)
        y = librosa.effects.time_stretch(y=y, rate=rate)
    elif aug_type == "pitch":
        n_steps = random.randint(-2, 2)
        y = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)
    elif aug_type == "noise":
        noise = np.random.randn(len(y))
        y = y + 0.005 * noise
    
    return y


def prepare_dataset(path, batch, apply_augment=False):
    speech_array, _ = librosa.load(path+'wavs/'+batch['audio_file'], sr=16000)
    
    if apply_augment:
        speech_array = augment_audio(speech_array, sr=16000)
    
    batch["input_features"] = feature_extractor(speech_array, sampling_rate=16000).input_features[0]
    return batch


stt_train_augmented = stt_train_dataset.map(lambda batch: prepare_dataset(path=stt_path + 'train/', batch=batch, apply_augment=True))
stt_train_dataset = stt_train_dataset.map(lambda batch: prepare_dataset(path=stt_path + 'train/', batch=batch, apply_augment=False))
stt_train_dataset = concatenate_datasets([stt_train_dataset, stt_train_augmented])

stt_test_dataset = stt_test_dataset.map(lambda batch: prepare_dataset(path=stt_path + 'test/', batch=batch, apply_augment=False))



tts_train_augmented = tts_train_dataset.map(lambda batch: prepare_dataset(path=tts_path + 'train/', batch=batch, apply_augment=True))
tts_train_dataset = tts_train_dataset.map(lambda batch: prepare_dataset(path=tts_path + 'train/', batch=batch, apply_augment=False))
tts_train_dataset = concatenate_datasets([tts_train_dataset, tts_train_augmented])

tts_test_dataset = tts_test_dataset.map(lambda batch: prepare_dataset(path=tts_path + 'test/', batch=batch, apply_augment=False))


bonus_train_augmented = bonus_train_dataset.map(lambda batch: prepare_dataset(path=bonus_path + 'train/', batch=batch, apply_augment=True))
bonus_train_dataset = bonus_train_dataset.map(lambda batch: prepare_dataset(path=bonus_path + 'train/', batch=batch, apply_augment=False))
bonus_train_dataset = concatenate_datasets([bonus_train_dataset, bonus_train_augmented])

bonus_test_dataset = bonus_test_dataset.map(lambda batch: prepare_dataset(path=bonus_path + 'test/', batch=batch, apply_augment=False))

In [ ]:
def has_features(batch):
    result = []
    for f in batch.get("input_features", []):
        if f is None:
            result.append(False)
        else:
            result.append(np.isfinite(f).all())
    return result

stt_train_dataset = stt_train_dataset.filter(has_features, batched=True, num_proc=2)
stt_test_dataset = stt_test_dataset.filter(has_features, batched=True, num_proc=2)
tts_train_dataset = tts_train_dataset.filter(has_features, batched=True, num_proc=2)
tts_test_dataset = tts_test_dataset.filter(has_features, batched=True, num_proc=2)
bonus_train_dataset = bonus_train_dataset.filter(has_features, batched=True, num_proc=2)
bonus_test_dataset = bonus_test_dataset.filter(has_features, batched=True, num_proc=2)

In [ ]:
processor = WhisperProcessor.from_pretrained(model_name, language=model_language, task="transcribe")


stt_train_dataset = stt_train_dataset.map(lambda batch: {"labels": processor.tokenizer(batch['text']).input_ids}, batched=True, num_proc=2)
stt_test_dataset = stt_test_dataset.map(lambda batch: {"labels": processor.tokenizer(batch['text']).input_ids}, batched=True, num_proc=2)

stt_train_dataset.set_format(type='torch', columns=['input_features', 'labels'])
stt_test_dataset.set_format(type='torch', columns=['input_features', 'labels'])



tts_train_dataset = tts_train_dataset.map(lambda batch: {"labels": processor.tokenizer(batch['text']).input_ids}, batched=True, num_proc=2)
tts_test_dataset = tts_test_dataset.map(lambda batch: {"labels": processor.tokenizer(batch['text']).input_ids}, batched=True, num_proc=2)

tts_train_dataset.set_format(type='torch', columns=['input_features', 'labels'])
tts_test_dataset.set_format(type='torch', columns=['input_features', 'labels'])



bonus_train_dataset = bonus_train_dataset.map(lambda batch: {"labels": processor.tokenizer(batch['text']).input_ids}, batched=True, num_proc=2)
bonus_test_dataset = bonus_test_dataset.map(lambda batch: {"labels": processor.tokenizer(batch['text']).input_ids}, batched=True, num_proc=2)

bonus_train_dataset.set_format(type='torch', columns=['input_features', 'labels'])
bonus_test_dataset.set_format(type='torch', columns=['input_features', 'labels'])


def non_empty_labels(batch):
    return [
        len(lbl) > 0
        for lbl in batch.get("labels", [])
    ]

stt_train_dataset = stt_train_dataset.filter(non_empty_labels, batched=True, num_proc=2)
stt_test_dataset  = stt_test_dataset.filter(non_empty_labels, batched=True, num_proc=2)
tts_train_dataset = tts_train_dataset.filter(non_empty_labels, batched=True, num_proc=2)
tts_test_dataset  = tts_test_dataset.filter(non_empty_labels, batched=True, num_proc=2)
bonus_train_dataset = bonus_train_dataset.filter(non_empty_labels, batched=True, num_proc=2)
bonus_test_dataset  = bonus_test_dataset.filter(non_empty_labels, batched=True, num_proc=2)


dataset = DatasetDict({
    "train": concatenate_datasets([stt_train_dataset, tts_train_dataset, bonus_train_dataset]),
    "test": concatenate_datasets([stt_test_dataset, tts_test_dataset, bonus_test_dataset]),
})

dataset

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_features": feature["input_features"]} for feature in features
        ]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        batch["attention_mask"] = torch.ones(batch["input_features"].shape[:-1], dtype=torch.long)


        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch
    
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
import evaluate
import re
import unicodedata

def normalize(text: str) -> str:
    text = text.lower()
    
    normalized_chars = []
    for ch in text:
        cat = unicodedata.category(ch)
        if cat.startswith("L") or cat == "Zs":
            normalized_chars.append(ch)
        elif unicodedata.combining(ch) and ch in "̄":
            normalized_chars.append(ch)
        elif ch in ["ӈ"]:
            normalized_chars.append(ch)

    return re.sub(r"\s+", " ", "".join(normalized_chars)).strip()

metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    orh_wer = 100 * metric_wer.compute(predictions=pred_str, references=label_str)
    orh_cer = 100 * metric_cer.compute(predictions=pred_str, references=label_str)

    pred_str_norm = [normalize(pred) for pred in pred_str]
    label_str_norm = [normalize(label) for label in label_str]

    pred_str_norm = [
        pred_str_norm[i] for i in range(len(pred_str_norm)) if len(label_str_norm[i]) > 0
    ]
    label_str_norm = [ 
        label_str_norm[i] for i in range(len(label_str_norm))if len(label_str_norm[i]) > 0
    ]

    wer = 100 * metric_wer.compute(predictions=pred_str_norm, references=label_str_norm)
    cer = 100 * metric_cer.compute(predictions=pred_str_norm, references=label_str_norm)

    return {"wer": wer, "cer": cer, "orh_wer": orh_wer, "orh_cer": orh_cer}

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.forced_bos_token_id = None

model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens = []
model.generation_config.forced_bos_token_id = None

model.config.apply_spec_augment = True
model.config.mask_time_prob = 0.05
model.config.mask_feature_prob = 0.05

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper_small_mansi_v3",
    overwrite_output_dir=True,

    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    warmup_ratio=0.05,
    
    group_by_length=True,
    gradient_checkpointing=False,
    fp16=False,
    fp16_full_eval=False,
    gradient_accumulation_steps=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=20,
    eval_strategy="steps",
    eval_steps=5000,
    save_strategy="steps",
    save_steps=5000,
    save_total_limit=2,
    
    seed=main_seed,
    data_seed=main_seed,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    report_to=["tensorboard"],
    push_to_hub=False,
    hub_private_repo=False,
    hub_always_push=False,
    predict_with_generate=True,
    generation_max_length=225
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)
trainer.train()

trainer.save_model("./whisper_small_mansi_v3")
processor.save_pretrained("./whisper_small_mansi_v3")